In [1]:
# # Dance Knowledge Graph Ontology
# Models the JSON schema as RDF/OWL where requested fields are represented as classes
# and values are modeled as named instances (resources with `schema:name`).


In [2]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, Literal
from rdflib.namespace import DCTERMS, SKOS

# Namespaces
SCHEMA = Namespace("http://schema.org/")
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
DANCE = Namespace("http://example.org/dance/")

# Graph setup
g = Graph()
g.bind("schema", SCHEMA)
g.bind("wd", WD)
g.bind("wdt", WDT)
g.bind("dance", DANCE)
g.bind("dcterms", DCTERMS)
g.bind("skos", SKOS)
g.bind("owl", OWL)



In [3]:
# Helpers

def add_class(uri, label, parent=None, comment=None):
    g.add((uri, RDF.type, OWL.Class))
    g.add((uri, RDFS.label, Literal(label, lang="en")))
    if parent is not None:
        g.add((uri, RDFS.subClassOf, parent))
    if comment is not None:
        g.add((uri, RDFS.comment, Literal(comment, lang="en")))


def add_obj_prop(uri, label, domain, range_):
    g.add((uri, RDF.type, OWL.ObjectProperty))
    g.add((uri, RDFS.label, Literal(label, lang="en")))
    g.add((uri, RDFS.domain, domain))
    g.add((uri, RDFS.range, range_))
    #if superprop is not None:
    #    g.add((uri, RDFS.subPropertyOf, superprop))
    #if equiv is not None:
     #   g.add((uri, OWL.equivalentProperty, equiv))


def add_data_prop(uri, label, domain, range_=XSD.string):
    g.add((uri, RDF.type, OWL.DatatypeProperty))
    g.add((uri, RDFS.label, Literal(label, lang="en")))
    g.add((uri, RDFS.domain, domain))
    g.add((uri, RDFS.range, range_))
    #if superprop is not None:
    #    g.add((uri, RDFS.subPropertyOf, superprop))


def add_named_instance(instance_uri, class_uri, name):
    g.add((instance_uri, RDF.type, class_uri))
    g.add((instance_uri, SCHEMA.name, Literal(name, lang="en")))



In [4]:
# ## Core classes (matching your JSON schema)


In [5]:
# Main record class (one row in your dataset)
add_class(
    DANCE.DanceRecord,
    "Dance Record",
    parent=SCHEMA.CreativeWork,
    comment="One record representing one dance style row from the dataset.",
)

# Requested "each item a class with name" classes
add_class(DANCE.DanceType, "Dance Type", parent=SKOS.Concept)
add_class(DANCE.DanceStyle, "Dance Style", parent=SKOS.Concept)
add_class(DANCE.Origin, "Origin", parent=SCHEMA.Place)
add_class(DANCE.TimePeriod, "Time Period", parent=SCHEMA.Intangible)
add_class(DANCE.Instrument, "Instrument", parent=SCHEMA.MusicInstrument)
add_class(DANCE.DanceFormation, "Dance Formation", parent=SKOS.Concept)
add_class(DANCE.Practitioner, "Practitioner", parent=SCHEMA.Person)
add_class(DANCE.EventFestival, "Event or Festival", parent=SCHEMA.Event)
add_class(DANCE.MusicGenre, "Music Genre", parent=SCHEMA.MusicGenre)
add_class(DANCE.HealthBenefit, "Health Benefit", parent=SCHEMA.MedicalEntity)
add_class(DANCE.AgeGroup, "Age Group", parent=SKOS.Concept)
add_class(DANCE.LearningDifficulty, "Learning Difficulty", parent=SKOS.Concept)


In [6]:
# ## Properties (aligned to schema fields)


In [7]:
# Object properties for class-based fields
add_obj_prop(DANCE.hasDanceType, "has dance type", DANCE.DanceRecord, DANCE.DanceType)
add_obj_prop(DANCE.hasDanceStyle, "has dance style", DANCE.DanceRecord, DANCE.DanceStyle)
add_obj_prop(DANCE.hasOrigin, "has origin", DANCE.DanceRecord, DANCE.Origin)
add_obj_prop(DANCE.hasTimePeriod, "has time period", DANCE.DanceRecord, DANCE.TimePeriod)
add_obj_prop(DANCE.hasInstrument, "has instrument", DANCE.DanceRecord, DANCE.Instrument)
add_obj_prop(DANCE.hasDanceFormation, "has dance formation", DANCE.DanceRecord, DANCE.DanceFormation)
add_obj_prop(DANCE.hasFamousPractitioner, "has famous practitioner", DANCE.DanceRecord, DANCE.Practitioner)
add_obj_prop(DANCE.hasEventFestival, "has event and festival", DANCE.DanceRecord, DANCE.EventFestival)
add_obj_prop(DANCE.hasAssociatedMusicGenre, "has associated music genre", DANCE.DanceRecord, DANCE.MusicGenre)
add_obj_prop(DANCE.hasHealthBenefit, "has health benefit", DANCE.DanceRecord, DANCE.HealthBenefit)
add_obj_prop(DANCE.hasAgeGroup, "has age group", DANCE.DanceRecord, DANCE.AgeGroup)
add_obj_prop(DANCE.hasLearningDifficulty, "has learning difficulty", DANCE.DanceRecord, DANCE.LearningDifficulty)
add_obj_prop(DANCE.hasTempoRange, "has tempo range", DANCE.DanceRecord, DANCE.TempoRange)

# Datatype properties for literal fields
add_data_prop(DANCE.culturalSignificance, "cultural significance", DANCE.DanceRecord, XSD.string)
add_data_prop(DANCE.notableCharacteristics, "notable characteristics", DANCE.DanceRecord, XSD.string)
add_data_prop(DANCE.hardnessRatio, "hardness ratio", DANCE.DanceRecord, XSD.decimal)
add_data_prop(DANCE.costume, "costume", DANCE.DanceRecord, XSD.string)
add_data_prop(DANCE.modernAdaptations, "modern adaptations", DANCE.DanceRecord, XSD.string)

# Tempo sub-properties for object {min, max}
add_data_prop(DANCE.minTempoBPM, "min tempo bpm", DANCE.TempoRange, XSD.integer)
add_data_prop(DANCE.maxTempoBPM, "max tempo bpm", DANCE.TempoRange, XSD.integer)



In [8]:
# ## Controlled vocab instances for enum-like fields


In [9]:
# dance_formation enum values
for value in ["Solo", "Partner", "Group", "Circle", "Line", "Square", "Freestyle", "Processional", "Mixed"]:
    add_named_instance(DANCE[f"formation_{value.replace(' ', '_')}"], DANCE.DanceFormation, value)

# learning_difficulty enum values
for value in ["Beginner", "Intermediate", "Advanced"]:
    add_named_instance(DANCE[f"difficulty_{value}"], DANCE.LearningDifficulty, value)

# age_group enum values
for value in ["Kids", "Teens", "Adults", "Seniors", "All ages"]:
    add_named_instance(DANCE[f"age_{value.replace(' ', '_')}"], DANCE.AgeGroup, value)



In [10]:
# add yt ontology
add_class(DANCE.YTLINK,"has corresponding yt video", parent=SCHEMA.VideoObject)
add_obj_prop(DANCE.hasYTLink, "has yt link", DANCE.DanceRecord, DANCE.hasYTLink)


add_data_prop(SCHEMA.description, "video description", DANCE.YTLINK, XSD.string)
add_data_prop(WD.Q28659447, "Sentiment", DANCE.YTLINK, XSD.string)



In [11]:
output_file = "dance_ontology.ttl"
g.serialize(destination=output_file, format="turtle")
print(f"Saved ontology to {output_file}")
print(f"Total triples: {len(g)}")

# Print a small sample
for i, triple in enumerate(g):
    if i >= 12:
        break
    print(triple)


Saved ontology to dance_ontology.ttl
Total triples: 169
(rdflib.term.URIRef('http://example.org/dance/notableCharacteristics'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('notable characteristics', lang='en'))
(rdflib.term.URIRef('http://example.org/dance/hasDanceStyle'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#ObjectProperty'))
(rdflib.term.URIRef('http://schema.org/description'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('video description', lang='en'))
(rdflib.term.URIRef('http://example.org/dance/DanceType'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#Class'))
(rdflib.term.URIRef('http://example.org/dance/hasDanceStyle'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#domain'), rdflib.term.URIRef('http://example.org/dance/DanceRec